## Assignment 2

Hi I just wanted to say that I tried to change the horoscope chat to a weather service but ran out of time, in here I managed to make it work, I will keep trying to make the chat work in the coming days

In [ ]:
import gradio as gr
import openai
import requests
import os
import numpy as np
import pandas as pd
import datetime
import json

In [29]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

import os
from openai import OpenAI   
client = OpenAI()

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [30]:

# dictionary for some cities
CITY_COORDS = {
    "toronto": (43.7, -79.38),
    "vancouver": (49.28, -123.12),
    "montreal": (45.50, -73.56),
    "colima": (19.24, 103.72),
    "paris": (48.85, 2.35)
}

#Service 1

In [31]:
def extract_city(user_input):
    input_lower = user_input.lower()
    for city in CITY_COORDS.keys():
        if city in input_lower:
            return city
    return None

def get_weather_summary(city: str):
    city_lower = city.lower()
    if city_lower not in CITY_COORDS:
        return f"Sorry, I only know a few cities right now: {', '.join(CITY_COORDS.keys())}."

    lat, lon = CITY_COORDS[city_lower]
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}&current_weather=true"
    )
    r = requests.get(url)
    if r.status_code != 200:
        return f"Couldn't fetch weather data for {city}."

    data = r.json().get("current_weather", {})
    temp = data.get("temperature")
    wind = data.get("windspeed")
    weather_info = f"Temperature {temp}°C and wind speed {wind} km/h in {city.title()}."

    # Prepare messages for chat in a specific tone llike in past assignment
    messages = [
        {"role": "system", "content": "You are a friendly rapper assistant giving short weather updates."},
        {"role": "user", "content": f"Summarize the following weather info for a friendly chat response:\n{weather_info}\n Make it with a mexican accent."}
    ]

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7
    )

    summary = completion.choices[0].message.content.strip()
    return summary

In [32]:
print(get_weather_summary("Colima"))

¡Hola, amigo! 🌞 In Colima right now, it’s a nice 19.4°C, feeling pretty chill, ya know? And the wind’s blowing softly at 3.7 km/h. Perfect day to vibe outside! ¡Disfruta! 🌴🎶
